# 1주차 · 거래량 시계열 예측 (ARIMA)

애플(AAPL) 주식의 **거래량(Volume)**을 예측합니다. 가격이 아니라 거래량인 이유 — 거래량은 요일/주기 패턴이 있어서 시계열 모델이 실력을 발휘할 수 있기 때문이에요.

**진행 방법**
1. 셀을 위에서부터 순서대로 실행하세요.
2. `✏️ 여기를 채우세요` 표시된 셀만 수정하면 됩니다.
3. 마지막 셀을 실행하면 `submission.csv`가 다운로드됩니다 → 웹 Submit 탭에 올리세요.

평가 지표: **RMSE** (낮을수록 좋음)

## 1. 데이터 불러오기  *(그냥 실행하세요)*

In [ ]:
import pandas as pd

# 웹 Data 탭에서 받은 파일의 공개 링크 (운영진이 채워둡니다)
TRAIN_URL = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/finance/train.csv"  # 예: https://raw.githubusercontent.com/.../train.csv
TEST_URL  = "https://raw.githubusercontent.com/nepersoned/dat-datasets/main/finance/test.csv"

if TRAIN_URL:
    train = pd.read_csv(TRAIN_URL, parse_dates=["date"])
    test  = pd.read_csv(TEST_URL,  parse_dates=["date"])
else:
    # URL이 없으면 파일을 직접 업로드하세요 (train.csv, test.csv)
    from google.colab import files
    up = files.upload()
    train = pd.read_csv("train.csv", parse_dates=["date"])
    test  = pd.read_csv("test.csv",  parse_dates=["date"])

print("train:", train.shape, "| test:", test.shape)
train.head()

## 2. 데이터 살펴보기  *(그냥 실행하세요)*
거래량 흐름과 **요일별 패턴**을 확인해봅니다. 계절성이 보이면 모델에 활용할 수 있어요.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(train["date"], train["volume"]) ; ax[0].set_title("거래량 추이")
train.groupby(train["date"].dt.dayofweek)["volume"].mean().plot.bar(ax=ax[1])
ax[1].set_title("요일별 평균 거래량 (0=월)")
plt.show()

## 3. ✏️ 여기를 채우세요 — 모델 학습

아래는 **동작하는 기본 예시(ARIMA)** 입니다. 그대로 제출해도 나이브 베이스라인은 이깁니다.

점수를 올리려면 이 부분을 바꿔보세요:
- `order=(p, d, q)` 값 조정
- 주간 계절성을 반영한 `SARIMAX` 사용 (`seasonal_order=(P,D,Q,5)`)
- 로그 변환 후 예측

In [ ]:
import numpy as np
from statsmodels.tsa.arima.model import ARIMA

# 거래량은 값이 커서 로그로 변환하면 예측이 안정적입니다
y = np.log(train["volume"].astype(float).values)

# ✏️ 이 order 값을 바꿔가며 실험해보세요 (지금 값도 나이브 베이스라인은 이깁니다)
model = ARIMA(y, order=(2, 0, 2))
fitted = model.fit()
print(fitted.summary().tables[0])

## 4. test 구간 예측  *(그냥 실행하세요)*

In [ ]:
pred = np.exp(fitted.forecast(steps=len(test)))  # 로그를 원래 스케일로 되돌림
test["prediction"] = pred
test[["id", "date", "prediction"]].head()

## 5. 제출 파일 저장  *(실행하면 자동 다운로드)*
다운로드된 `submission.csv`를 웹 **Submit 탭**에 업로드하세요.

In [ ]:
submission = test[["id", "prediction"]]
submission.to_csv("submission.csv", index=False)
print("저장 완료:", submission.shape)

try:
    from google.colab import files
    files.download("submission.csv")
except Exception:
    print("코랩이 아니면 왼쪽 파일탭에서 submission.csv를 내려받으세요.")